# Stage 3: Post-processing dataset Image Captioning (Vietnamese)

Notebook này đọc CSV được xuất ra từ table `dataset` trên Supabase và xuất dataset cuối cùng theo schema:

- json (`dataset_final.json`) hoặc jsonl (`dataset_final.jsonl`)
- Trường bắt buộc: `id`, `metadata`, `image`, `caption_short`, `caption_detail`  


## 0. Schema output mong muốn

```json
{
  "id": "SGK_CanhDieu_DaoDuc_1_page_001",
  "image": "SGK_CanhDieu_DaoDuc_1_page_001.png",
  "metadata": {
    "Type": "SGK",
    "Collection": "Canh Dieu",
    "Title": "Dao Duc 1",
    "Author": "",
    "Publisher": ""
  },
  "caption_short": "Mô tả ngắn…",
  "caption_detail": "Mô tả chi tiết…",
}
```


## 1. CSV Supabase có gì?

Trong file csv input, ngoài các cột caption còn có:

- `page_type`, `has_text`, `has_table`
- `objects_present` (list JSON nhưng nằm trong 1 ô CSV dạng chuỗi)
- `auto_flags`, `error_tags` (list JSON dạng chuỗi)
- `is_checked`, `review_priority`, `notes`
- `created_at`, `updated_at`

Notebook này **chỉ lấy các cột cần để export final**, còn các cột khác dùng để tạo QC report.


## 2. Cấu hình

In [ ]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

INPUT_SUPABASE_CSV = Path("../data/stage3_inputs/supabase_export_full.csv")
IMAGES_DIR = Path("../data/processed")
OUT_DIR = Path("../output/stage3_processed_dataset")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Metadata: Mỗi row trong metadata_catalog.csv tương ứng 1 quyển sách, định danh bằng cột `prefix`.
METADATA_CATALOG = Path("../data/stage3_inputs/metadata_catalog.csv")

# Nếu STRICT_METADATA = True: id nào không match prefix sẽ raise error để bắt lỗi sớm.
STRICT_METADATA = True

## 4. Hàm parse JSON cell trong CSV

Vì CSV không lưu list/object native, Supabase sẽ export list JSON như:
- `["a","b"]`
- `[]`

Ta cần parse lại để QC/logic không bị sai.


In [ ]:
RE_WHITESPACE = re.compile(r"\s+")
RE_JSON_LIKE = re.compile(r"^\s*(\[.*\]|\{.*\})\s*$", re.DOTALL)

def clean_text(s: Any) -> Optional[str]:
    if s is None:
        return None
    if isinstance(s, float) and math.isnan(s):
        return None
    s = str(s).strip()
    s = RE_WHITESPACE.sub(" ", s)
    return s or None

def parse_json_cell(val: Any, default):
    """
    Parse list/dict được export trong 1 ô CSV.
    Nếu không parse được thì trả default.
    """
    if val is None:
        return default
    if isinstance(val, float) and math.isnan(val):
        return default
    if isinstance(val, (list, dict)):
        return val
    s = str(val).strip()
    if s == "":
        return default
    if not RE_JSON_LIKE.match(s):
        return default
    try:
        return json.loads(s)
    except Exception:
        return default

def ensure_image_filename(img: str) -> str:
    # Nếu muốn giữ cả path tương đối thì bỏ .name
    return Path(img).name


## 5. Load CSV từ Supabase

In [ ]:
df = pd.read_csv(INPUT_SUPABASE_CSV)
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()

## 6. Convert row → schema final

Mapping:
- `id` <- cột `id`
- `image` <- cột `image`
- `caption_short` <- cột `caption_short`
- `caption_detail` <- cột `caption_detail`
- `metadata` <- lookup theo catalog


In [ ]:
# Load metadata catalog (prefix -> metadata)
if not METADATA_CATALOG.exists():
    raise FileNotFoundError(f"metadata catalog not found: {METADATA_CATALOG.resolve()}")

catalog_df = pd.read_csv(METADATA_CATALOG)

# Validate required column
if "prefix" not in catalog_df.columns:
    raise ValueError("metadata_catalog.csv must contain a 'prefix' column")

# Normalize + sort prefixes (longest first) to avoid short-prefix mismatches
catalog_df["prefix"] = catalog_df["prefix"].astype(str)
catalog_df = catalog_df.sort_values(by="prefix", key=lambda s: s.str.len(), ascending=False)

CATALOG_ENTRIES: List[tuple[str, Dict[str, Any]]] = []
for _, r in catalog_df.iterrows():
    entry = r.to_dict()
    prefix = str(entry.pop("prefix")).strip()
    if not prefix:
        continue

    # drop NaN / empty strings
    meta = {k: v for k, v in entry.items() if pd.notna(v) and str(v).strip() != ""}

    # cast Grade if present
    if "Grade" in meta:
        try:
            meta["Grade"] = int(meta["Grade"])
        except Exception:
            # keep as-is if cannot cast
            pass

    CATALOG_ENTRIES.append((prefix, meta))

if len(CATALOG_ENTRIES) == 0:
    raise ValueError("metadata_catalog.csv is empty (no valid prefix rows found)")

def build_metadata_from_catalog(sample_id: str) -> Dict[str, Any]:
    sample_id = (sample_id or "").strip()
    if not sample_id:
        if STRICT_METADATA:
            raise ValueError("Empty id encountered while building metadata")
        return {}

    for prefix, meta in CATALOG_ENTRIES:
        if sample_id.startswith(prefix):
            # return a copy to avoid accidental mutation
            return dict(meta)

    if STRICT_METADATA:
        raise ValueError(
            f"No metadata_catalog match for id='{sample_id}'. "
            f"Please add a row with a matching prefix in {METADATA_CATALOG}."
        )
    return {}

def build_metadata(row: pd.Series) -> Dict[str, Any]:
    return build_metadata_from_catalog(str(row.get("id", "")))

def convert_row(row: pd.Series) -> Dict[str, Any]:
    out = {
        "id": clean_text(row.get("id")),
        "image": ensure_image_filename(clean_text(row.get("image")) or ""),
        "metadata": build_metadata(row),
        "caption_short": clean_text(row.get("caption_short")),
        "caption_detail": clean_text(row.get("caption_detail")),
    }
    return out

converted = [convert_row(r) for _, r in df.iterrows()]
converted[:1]


## 7. Export JSON / JSONL

In [ ]:
EXPORT_ONLY_CHECKED = False
ALLOW_STATUSES = {"checked"}

final_rows = []
for (_, row), out in zip(df.iterrows(), converted):
    status = clean_text(row.get("is_checked")) or ""
    if EXPORT_ONLY_CHECKED and status not in ALLOW_STATUSES:
        continue
    final_rows.append(out)

print("Final rows:", len(final_rows))


In [ ]:
out_json = OUT_DIR / "dataset_final.json"
out_jsonl = OUT_DIR / "dataset_final.jsonl"

with out_json.open("w", encoding="utf-8") as f:
    json.dump(final_rows, f, ensure_ascii=False, indent=2)

with out_jsonl.open("w", encoding="utf-8") as f:
    for r in final_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Wrote:", out_json)
print("Wrote:", out_jsonl)


## 8. (Tuỳ chọn) Split train/val/test

Nếu cần, có thể chia 80/10/10.  


In [ ]:
import random

def make_splits(items: List[Dict[str, Any]], seed: int = 42, ratios=(0.8, 0.1, 0.1)):
    assert abs(sum(ratios) - 1.0) < 1e-6
    items = list(items)
    random.Random(seed).shuffle(items)
    n = len(items)
    n_train = int(n * ratios[0])
    n_val = int(n * ratios[1])
    train = items[:n_train]
    val = items[n_train:n_train+n_val]
    test = items[n_train+n_val:]
    return train, val, test

# train, val, test = make_splits(final_rows)
# for name, part in [("train", train), ("val", val), ("test", test)]:
#     p = OUT_DIR / f"dataset_{name}.jsonl"
#     with p.open("w", encoding="utf-8") as f:
#         for r in part:
#             f.write(json.dumps(r, ensure_ascii=False) + "\n")
#     print(name, len(part), "->", p)
